# Monte Carlo Simulation with Python
### Practical Business Python - February 2019

https://pbpython.com/monte-carlo.html

### Introduction
There are many sophisticated models people can build for solving a forecasting problem. However, they frequently stick to simple Excel models based on average historical values, intuition and some high level domain-specific heuristics. This approach may be precise enough for the problem at hand but there are alternatives that can add more information to the prediction with a reasonable amount of additional effort.

One approach that can produce a better understanding of the range of potential outcomes and help avoid the “flaw of averages” is a Monte Carlo simulation. The rest of **this article will describe how to use python with pandas and numpy to build a Monte Carlo simulation to predict the range of potential values for a sales compensation budget**. This approach is meant to be simple enough that it can be used for other problems you might encounter but also powerful enough to provide insights that a basic “gut-feel” model can not provide on its own.

### Problem Background
For this example, we will try to predict how much money we should budget for sales commissions for the next year. This problem is useful for modeling because we have a defined formula for calculating commissions and we likely have some experience with prior years’ commissions payments.

This problem is also important from a business perspective. Sales commissions can be a large selling expense and it is important to plan appropriately for this expense. In addition, the use of a Monte Carlo simulation is a relatively simple improvement that can be made to augment what is normally an unsophisticated estimation process.

In this example, the sample sales commission would look like this for a 5 person sales force:

In this example, the commission is a result of this formula:

Commission Amount = Actual Sales * Commission Rate

The commission rate is based on this Percent To Plan table:

Before we build a model and run the simulation, let’s look at a simple approach for predicting next year’s commission expense.

### Naïve Approach to the Problem
Imagine your task as Amy or Andy analyst is to tell finance how much to budget for sales commissions for next year. One approach might be to assume everyone makes 100% of their target and earns the 4% commission rate. Plugging these values into Excel yields this:

Sample Commissions Calc
Imagine you present this to finance, and they say, “We never have everyone get the same commission rate. We need a more accurate model.”

For round two, you might try a couple of ranges:



Now, you have a little bit more information and go back to finance. This time finance says, “this range is useful but what is your confidence in this range? Also, we need you to do this for a sales force of 500 people and model several different rates to determine the amount to budget.” Hmmm… Now, what do you do?

This simple approach illustrates the basic iterative method for a Monte Carlo simulation. You iterate through this process many times in order to determine a range of potential commission values for the year. Doing this manually by hand is challenging. Fortunately, python makes this approach much simpler.

### Monte Carlo
Now that we have covered the problem at a high level, we can discuss how Monte Carlo analysis might be a useful tool for predicting commissions expenses for the next year. At its simplest level, a Monte Carlo analysis (or simulation) involves running many scenarios with different random inputs and summarizing the distribution of the results.

Using the commissions analysis, we can continue the manual process we started above but run the program 100’s or even 1000’s of times and we will get a distribution of potential commission amounts. This distribution can inform the likelihood that the expense will be within a certain window. At the end of the day, this is a prediction so we will likely never predict it exactly. We can develop a more informed idea about the potential risk of under or over budgeting.

There are two components to running a Monte Carlo simulation:

the equation to evaluate
the random variables for the input
We have already described the equation above. Now we need to think about how to populate the random variables.

One simple approach would be to take a random number between 0% and 200% (representing our intuition about commissions rates). However, because we pay commissions every year, we understand our problem in a little more detail and can use that prior knowledge to build a more accurate model.

Because we have paid out commissions for several years, we can look at a typical historical distribution of percent to target:

### Building a Python Model
We can use pandas to construct a model that replicates the Excel spreadsheet calculation. There are other python approaches to building Monte Carlo models but I find that this pandas method is conceptually easier to comprehend if you are coming from an Excel background. It also has the added benefit of generating pandas dataframes that can be inspected and reviewed for reasonableness.

First complete our imports and set our plotting style:

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

sns.set_style('whitegrid')

For this model, we will use a random number generation from numpy. The handy aspect of numpy is that there are several random number generators that can create random samples based on a predefined distribution.

As described above, we know that our historical percent to target performance is centered around a a mean of 100% and standard deviation of 10%. Let’s define those variables as well as the number of sales reps and simulations we are modeling:

For this model, we will use a random number generation from numpy. The handy aspect of numpy is that there are several random number generators that can create random samples based on a predefined distribution.

As described above, we know that our historical percent to target performance is centered around a a mean of 100% and standard deviation of 10%. Let’s define those variables as well as the number of sales reps and simulations we are modeling:

In [2]:
avg = 1
std_dev = .1
num_reps = 500
num_simulations = 1000

Now we can use numpy to generate a list of percentages that will replicate our historical normal distribution:

In [3]:
pct_to_target = np.random.normal(avg, std_dev, num_reps).round(2)
pct_to_target

array([1.05, 0.96, 0.85, 1.02, 1.02, 1.2 , 1.13, 1.05, 1.17, 0.86, 0.99,
       0.84, 0.88, 0.77, 1.  , 1.05, 0.97, 0.96, 0.92, 0.99, 1.  , 1.06,
       1.08, 0.97, 0.97, 0.93, 1.  , 1.03, 0.99, 1.09, 0.98, 0.94, 1.1 ,
       0.93, 1.05, 0.95, 0.96, 0.95, 1.09, 1.05, 1.  , 0.96, 1.1 , 0.96,
       0.94, 1.  , 0.79, 0.92, 0.84, 0.93, 0.91, 0.92, 1.07, 0.9 , 1.04,
       0.99, 0.96, 1.01, 1.06, 0.98, 0.88, 0.92, 1.02, 1.03, 0.93, 1.07,
       0.91, 0.95, 1.09, 0.87, 0.99, 0.86, 0.9 , 1.05, 1.09, 1.02, 0.78,
       0.92, 0.92, 0.94, 1.07, 1.07, 1.23, 1.01, 0.95, 1.13, 0.99, 1.08,
       1.04, 1.06, 0.88, 0.95, 1.11, 1.  , 1.12, 0.95, 0.92, 0.87, 0.92,
       1.12, 1.16, 1.  , 0.79, 0.83, 0.71, 0.97, 1.04, 1.15, 1.02, 1.1 ,
       1.05, 0.98, 0.88, 1.21, 1.12, 0.88, 1.19, 0.88, 1.02, 1.12, 1.09,
       0.91, 1.01, 1.02, 1.21, 1.08, 1.14, 0.98, 1.01, 0.92, 1.08, 0.99,
       1.05, 1.16, 0.83, 1.05, 1.05, 1.  , 1.1 , 1.04, 0.84, 1.04, 0.89,
       1.04, 1.05, 0.95, 1.03, 1.14, 0.99, 0.78, 0.

For this example, I have chosen to round it to 2 decimal places in order to make it very easy to see the boundaries.

Here is what the first 10 items look like:

array([0.92, 0.98, 1.1 , 0.93, 0.92, 0.99, 1.14, 1.28, 0.91, 1.  ])

This is a good quick check to make sure the ranges are within expectations.

Since we are trying to make an improvement on our simple approach, we are going to stick with a normal distribution for the percent to target. By using numpy though, we can adjust and use other distribution for future models if we must. However, I do warn that you should not use other models without truly understanding them and how they apply to your situation.

There is one other value that we need to simulate and that is the actual sales target. In order to illustrate a different distribution, we are going to assume that our sales target distribution looks something like this:

This is definitely not a normal distribution. This distribution shows us that sales targets are set into 1 of 6 buckets and the frequency gets lower as the amount increases. This distribution could be indicative of a very simple target setting process where individuals are bucketed into certain groups and given targets consistently based on their tenure, territory size or sales pipeline.

For the sake of this example, we will use a uniform distribution but assign lower probability rates for some of the values.

Here is how we can build this using numpy.random.choice

In [4]:
sales_target_values = [75_000, 100_000, 200_000, 300_000, 400_000, 500_000]
sales_target_prob = [.3, .3, .2, .1, .05, .05]
sales_target = np.random.choice(sales_target_values, num_reps, p = sales_target_prob)
sales_target

array([ 75000,  75000, 300000,  75000, 300000, 300000,  75000,  75000,
       200000,  75000,  75000,  75000, 200000,  75000, 400000, 200000,
       100000,  75000,  75000,  75000, 100000, 400000,  75000,  75000,
       400000, 200000, 400000,  75000, 400000, 200000,  75000, 500000,
        75000, 100000,  75000, 100000, 400000, 200000,  75000,  75000,
       200000,  75000, 100000, 100000, 100000, 200000, 100000,  75000,
       200000, 200000, 200000, 200000, 100000,  75000, 100000, 100000,
       100000,  75000, 200000, 200000, 200000, 100000,  75000, 100000,
       100000,  75000, 100000, 100000, 200000,  75000,  75000,  75000,
       200000,  75000, 100000, 200000, 200000, 100000, 200000, 100000,
       500000, 300000, 400000,  75000, 300000, 200000, 100000, 400000,
        75000,  75000, 300000, 500000, 100000, 500000, 400000,  75000,
       500000, 200000,  75000, 100000,  75000, 100000, 300000, 200000,
       300000, 200000, 200000, 200000, 200000, 100000, 200000,  75000,
      

Admittedly this is a somewhat contrived example but I wanted to show how different distributions could be incorporated into our model.

Now that we know how to create our two input distributions, let’s build up a pandas dataframe:

In [5]:
df = pd.DataFrame(index=range(num_reps), data={'Pct_To_Target': pct_to_target,
                                               'Sales_Target': sales_target})

df['Sales'] = df['Pct_To_Target'] * df['Sales_Target']

Here is what our new dataframe looks like:

In [6]:
df

,Pct_To_Target,Sales_Target,Sales
0,1.05,75000,78750.0
1,0.96,75000,72000.0
2,0.85,300000,255000.0
3,1.02,75000,76500.0
4,1.02,300000,306000.0
...,...,...,...
495,1.01,200000,202000.0
496,1.01,400000,404000.0
497,1.17,100000,117000.0
498,1.03,75000,77250.0


You might notice that I did a little trick to calculate the actual sales amount. For this problem, the actual sales amount may change greatly over the years but the performance distribution remains remarkably consistent. Therefore, I’m using the random distributions to generate my inputs and backing into the actual sales.

The final piece of code we need to create is a way to map our Pct_To_Target to the commission rate. Here is the function:

In [7]:
def calc_commission_rate(x):
    """ Return the commission rate based on the table:
    0   - 90%  = 2%
    91  - 99%  = 3%
      >= 100%  = 4%
    """
    if x <= .90:
        return .02
    if x <= .99:
        return .03
    else:
        return .04

The added benefit of using python instead of Excel is that we can create much more complex logic that is easier to understand than if we tried to build a complex nested if statement in Excel.

Now we create our commission rate and multiply it times sales:

In [8]:
df['Commission_Rate'] = df['Pct_To_Target'].apply(calc_commission_rate)
df['Commission_Amount'] = df['Commission_Rate'] * df['Sales']

Which yields this result, which looks very much like an Excel model we might build:

In [9]:
df

,Pct_To_Target,Sales_Target,Sales,Commission_Rate,Commission_Amount
0,1.05,75000,78750.0,0.04,3150.0
1,0.96,75000,72000.0,0.03,2160.0
2,0.85,300000,255000.0,0.02,5100.0
3,1.02,75000,76500.0,0.04,3060.0
4,1.02,300000,306000.0,0.04,12240.0
...,...,...,...,...,...
495,1.01,200000,202000.0,0.04,8080.0
496,1.01,400000,404000.0,0.04,16160.0
497,1.17,100000,117000.0,0.04,4680.0
498,1.03,75000,77250.0,0.04,3090.0


In [10]:
k = df.loc[0:5,['Commission_Amount']].sum(axis = 0)[0]
# https://www.geeksforgeeks.org/how-to-sum-values-of-pandas-dataframe-by-rows/
k

C:\Users\hilton.netta\AppData\Local\Temp\ipykernel_3096\1168147185.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  k = df.loc[0:5,['Commission_Amount']].sum(axis = 0)[0]


40110.0

There you have it!

We have replicated a model that is similar to what we would have done in Excel but we used some more sophisticated distributions than just throwing a bunch of random number inputs into the problem.

In [11]:
print(f'If we sum up the values (only the top 5 are shown above) in the Commission_Amount column, we can see that \
this simulation shows that we would pay {df.loc[0:5,["Commission_Amount"]].sum(axis = 0)[0]}')

If we sum up the values (only the top 5 are shown above) in the Commission_Amount column, we can see that this simulation shows that we would pay 40110.0


C:\Users\hilton.netta\AppData\Local\Temp\ipykernel_3096\3829305155.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  this simulation shows that we would pay {df.loc[0:5,["Commission_Amount"]].sum(axis = 0)[0]}')


### Let’s Loop
The real “magic” of the Monte Carlo simulation is that if we run a simulation many times, we start to develop a picture of the likely distribution of results. In Excel, you would need VBA or another plugin to run multiple iterations. In python, we can use a for loop to run as many simulations as we’d like.

In addition to running each simulation, we save the results we care about in a list that we will turn into a dataframe for further analysis of the distribution of results.

Here is the full for loop code:

In [12]:
# Define a list to keep all the results from each simulation that we want to analyze
all_stats = []

# Loop through many simulations
for i in range(num_simulations):

    # Choose random inputs for the sales targets and percent to target
    sales_target = np.random.choice(sales_target_values, num_reps, p=sales_target_prob)
    pct_to_target = np.random.normal(avg, std_dev, num_reps).round(2)

    # Build the dataframe based on the inputs and number of reps
    df = pd.DataFrame(index=range(num_reps), data={'Pct_To_Target': pct_to_target,
                                                   'Sales_Target': sales_target})

    # Back into the sales number using the percent to target rate
    df['Sales'] = df['Pct_To_Target'] * df['Sales_Target']

    # Determine the commissions rate and calculate it
    df['Commission_Rate'] = df['Pct_To_Target'].apply(calc_commission_rate)
    df['Commission_Amount'] = df['Commission_Rate'] * df['Sales']

    # We want to track sales,commission amounts and sales targets over all the simulations
    all_stats.append([df['Sales'].sum().round(0),
                      df['Commission_Amount'].sum().round(0),
                      df['Sales_Target'].sum().round(0)])

In [71]:
all_stats

[[84764750.0, 2899102.0, 84500000],
 [82975500.0, 2841912.0, 83100000],
 [81780500.0, 2841575.0, 81450000],
 [82893750.0, 2813985.0, 83075000],
 [84630750.0, 2946678.0, 84375000],
 [82019750.0, 2853730.0, 81550000],
 [79091250.0, 2668690.0, 79250000],
 [89202000.0, 3011440.0, 89850000],
 [85789250.0, 2911308.0, 85825000],
 [85820250.0, 2917978.0, 86250000],
 [81177250.0, 2803790.0, 80825000],
 [84411750.0, 2889620.0, 84575000],
 [82181000.0, 2787092.0, 82525000],
 [80927500.0, 2722652.0, 81275000],
 [84876750.0, 2855262.0, 85350000],
 [79010000.0, 2721455.0, 79000000],
 [85592000.0, 2893292.0, 85850000],
 [82156000.0, 2809948.0, 81900000],
 [88197500.0, 3055768.0, 87225000],
 [83538500.0, 2910402.0, 82900000],
 [79869250.0, 2717348.0, 79475000],
 [87622250.0, 3049500.0, 86875000],
 [82758500.0, 2789068.0, 82950000],
 [85244000.0, 2906530.0, 85375000],
 [79195500.0, 2675618.0, 79550000],
 [83024750.0, 2828058.0, 83325000],
 [84383750.0, 2927672.0, 83700000],
 [84666000.0, 2865788.0, 850

While this may seem a little intimidating at first, we are only including 7 python statements inside this loop that we can run as many times as we want. On my standard laptop, I can run 1000 simulations in 2.75s so there is no reason I can’t do this many more times if need be.

At some point, there are diminishing returns. The results of 1 Million simulations are not necessarily any more useful than 10,000. My advice is to try different amounts and see how the output changes.

In order to analyze the results of the simulation, I will build a dataframe from all_stats :

In [13]:
results_df = pd.DataFrame.from_records(all_stats, columns=['Sales',
                                                           'Commission_Amount',
                                                           'Sales_Target'])
results_df

,Sales,Commission_Amount,Sales_Target
0,85470500.0,2977448.0,85025000
1,82852000.0,2909612.0,81600000
2,83843250.0,2872275.0,83350000
3,82774000.0,2858772.0,82625000
4,85466500.0,2931318.0,85075000
...,...,...,...
995,87040750.0,2946008.0,87700000
996,79846500.0,2730485.0,79750000
997,83762500.0,2889580.0,83250000
998,83418500.0,2826305.0,83500000


Now, it is easy to see what the range of results look like:

In [74]:
results_df.describe().style.format('{:,}')

,Sales,Commission_Amount,Sales_Target
count,"1,000.0","1,000.0","1,000.0"
mean,"83,759,101.25","2,861,025.985","83,740,075.0"
std,"2,594,958.473953405","99,022.14848001624","2,551,225.856867194"
min,"75,133,250.0","2,548,982.0","75,350,000.0"
25%,"81,846,312.5","2,788,366.0","81,850,000.0"
50%,"83,822,625.0","2,864,880.0","83,750,000.0"
75%,"85,530,125.0","2,929,993.0","85,450,000.0"
max,"91,255,500.0","3,086,742.0","91,925,000.0"


So, what does this chart and the output of describe tell us? We can see that the average commissions expense is $2.85M and the standard deviation is $103K. We can also see that the commissions payment can be as low as $2.5M or as high as $3.2M.

Based on these results, how comfortable are you that the expense for commissions will be less than $3M? Or, if someone says, “Let’s only budget $2.7M” would you feel comfortable that your expenses would be below that amount? Probably not.

Therein lies one of the benefits of the Monte Carlo simulation. You develop a better understanding of the distribution of likely outcomes and can use that knowledge plus your business acumen to make an informed estimate.

The other value of this model is that you can model many different assumptions and see what happens. Here are some simple changes you can make to see how the results change:

Increase top commission rate to 5%
Decrease the number of sales people
Change the expected standard deviation to a higher amount
Modify the distribution of targets
Now that the model is created, making these changes is as simple as a few variable tweaks and re-running your code. You can view the notebook associated with this post on github.

Another observation about Monte Carlo simulations is that they are relatively easy to explain to the end user of the prediction. The person receiving this estimate may not have a deep mathematical background but can intuitively understand what this simulation is doing and how to assess the likelihood of the range of potential results.

Finally, I think the approach shown here with python is easier to understand and replicate than some of the Excel solutions you may encounter. Because python is a programming language, there is a linear flow to the calculations which you can follow.

### Conclusion
A Monte Carlo simulation is a useful tool for predicting future results by calculating a formula multiple times with different random inputs. This is a process you can execute in Excel but it is not simple to do without some VBA or potentially expensive third party plugins. Using numpy and pandas to build a model and generate multiple potential results and analyze them is relatively straightforward. The other added benefit is that analysts can run many scenarios by changing the inputs and can move on to much more sophisticated models in the future if the needs arise. Finally, the results can be shared with non-technical users and facilitate discussions around the uncertainty of the final results.

I hope this example is useful to you and gives you ideas that you can apply to your own problems. Please feel free to leave a comment if you find this article helpful for developing your own estimation models.

Updates
19-March-2019: Based on comments from reddit, I have made another implementation which is faster: https://github.com/chris1610/pbpython/blob/master/notebooks/Monte_Carlo_Simulationv2.ipynb